In [132]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.nn.functional import one_hot


In [74]:
class ModuloDataset(Dataset):
    def __init__(self, data):
        self.inputs = data[:,:3]
        self.labels = data[:,3:].squeeze(dim=1)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        return self.inputs[index], self.labels[index]

In [165]:
### Create dataset
modulo_number = 3
data = []

for i in range(modulo_number):
    for j in range(modulo_number):
        data.append([i, j, modulo_number, (i + j) % modulo_number])

data = torch.tensor(data)

In [83]:
train_perecentage = 0.3

shuffled_data = data[torch.randperm(data.shape[0])]
train_idx = int(train_perecentage * shuffled_data.shape[0])
train_split = shuffled_data[:train_idx]
test_split = shuffled_data[train_idx:]
print(train_split.shape[0], test_split.shape[0])

train_dataset = ModuloDataset(train_split)
test_dataset = ModuloDataset(test_split)

2 7


In [163]:
from torch.nn import Embedding, MultiheadAttention, Linear, Module, ReLU

class MLP(nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.up_projection = Linear(d_model, d_hidden)
        self.down_projection = Linear(d_hidden, d_model)
        self.relu = ReLU()

    def forward(self, x):
        h1 = self.up_projection(x)
        r1 = self.relu(h1)
        out = self.down_projection(r1)
        return out

class ModuloTransformer(Module):
    def __init__(self, modulo_number, d_embed, n_heads, d_ff):
        super().__init__()
        self.embed = Embedding(modulo_number + 1, d_embed)      # +1 because of the '=' token
        self.pos_embed = Embedding(3, d_embed)      # hard code to 3 bc seq len is always that
        self.mh_attn = MultiheadAttention(d_embed, n_heads, batch_first=True)     # not masked attention, batch is first dim
        self.mlp = MLP(d_embed, d_ff)
        self.unembed = Linear(d_embed, modulo_number)

    def forward(self, x):
        x_embed = self.embed(x)
        pos_embed = self.pos_embed(torch.arange(3))
        x_embed += pos_embed
        x_attn, _ = self.mh_attn(x_embed, x_embed, x_embed)
        x_attn += x_embed               # residual
        x_ff = x_attn + self.mlp(x_attn) # residual
        logits = self.unembed(x_ff)
        probs = torch.softmax(logits, dim=-1)
        return probs
    
    def predict(self, x):
        probs = self.forward(x)
        return probs[:, 2]

In [ ]:
m_t = ModuloTransformer(modulo_number, 128, 4, 512)



(tensor([[[0.2873, 0.4514, 0.2613],
          [0.3970, 0.1877, 0.4153],
          [0.3755, 0.2207, 0.4037]],
 
         [[0.2595, 0.5078, 0.2327],
          [0.3270, 0.5513, 0.1217],
          [0.3853, 0.2557, 0.3591]]], grad_fn=<SoftmaxBackward0>),
 tensor([[0.3755, 0.2207, 0.4037],
         [0.3853, 0.2557, 0.3591]], grad_fn=<SelectBackward0>))